# Chat.DT — local experiment runner (Outlines + HF, GPU)

Runs all four settings (`DIRECT_QA_BASELINE`, `DIRECT_QA_GROUNDED`, `CYPHER_SOFT`,
`CYPHER_STRICT`) from a Colab GPU runtime against a remote Chat.DT API.

**Architecture**

- **Server (`api` container)** owns Neo4j (Bolt is internal-only), the test set,
  pre-executed gold queries, Cypher execution, and SVR/SCR/EA scoring.
- **This notebook** owns the LLM. It loads the server-built `bundle_<model>.json`
  to rehydrate vocabulary + IDS schema + model dump, runs Outlines-constrained
  decoding locally on the GPU, and POSTs each generated output to `/evaluate`.

Bolt never leaves the Docker network. The LLM never leaves Colab.

**Inputs**

1. A server-built `bundle_<model>.json` (from `scripts/build_bundle.py`).
2. The HTTPS `API_BASE_URL` of your Chat.DT deployment (e.g. behind ngrok / Tailscale).
3. The `API_BEARER_TOKEN` configured server-side.
4. An HF model id or local checkpoint path (Outlines requires a local model for STRICT).

Use a T4 / L4 / A100 Colab runtime — torch + transformers + outlines are heavy.

## 1. Clone the repo and install GPU deps

In [ ]:
%%bash
set -e
if [ ! -d Chat.DT ]; then
  git clone --depth 1 https://github.com/IvansSmirnoff/Chat.DT.git
fi
cd Chat.DT
pip install -q -r requirements-base.txt
# Full LLM stack: torch, transformers, outlines, etc.
pip install -q -r requirements-llm.txt
python -c 'import torch; print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name() if torch.cuda.is_available() else "no gpu")'

In [ ]:
import os, sys
sys.path.insert(0, 'Chat.DT')

# --- Remote Chat.DT API (replace with your deployment) ---
os.environ['API_BASE_URL']     = 'https://YOUR_TUNNEL_HOST'   # e.g. https://chat-dt.ngrok.app
os.environ['API_BEARER_TOKEN'] = 'REPLACE_ME'                 # must match the server's API_BEARER_TOKEN

# --- LLM (runs in this Colab runtime) ---
os.environ['LLM_PROVIDER']    = 'local'
os.environ['LLM_MODEL_NAME']  = 'Qwen/Qwen2.5-7B-Instruct'    # HF repo id or /content/<path>

# Pydantic settings reads NEO4J_* with non-empty defaults; we never connect to
# Neo4j from Colab, but the import path still constructs Settings — leaving the
# defaults from src/config.py is fine.

## 2. Upload the bundle and ping the API

Drag `bundle_<model>.json` into the Colab file pane (or mount Drive). Then run a
readiness check against the API — bad token / unreachable Neo4j surfaces here
before any LLM work happens.

In [ ]:
from pathlib import Path
from src.client.api_client import ApiClient

BUNDLE_PATH = Path('bundle_barcelona.json')   # change to your bundle filename
assert BUNDLE_PATH.exists(), f'Bundle not found: {BUNDLE_PATH}. Upload it via the file pane.'

client = ApiClient(os.environ['API_BASE_URL'], os.environ['API_BEARER_TOKEN'])
print('health:', client.health())
print('ready: ', client.health_ready())
print('test cases on server:', len(client.get_test_set()))

## 3. Run all four settings via the API

The runner loads the bundle locally, builds the LLM engine on this GPU, and
streams `(question, predicted_output)` to `POST /evaluate` for scoring.

In [ ]:
from src.config import ExperimentSetting
from src.client.runner import ApiExperimentRunner, ApiRunnerConfig

config = ApiRunnerConfig(
    bundle_path=BUNDLE_PATH,
    output_dir=Path('results'),
    name='local_run',
    settings=list(ExperimentSetting),  # all 4
)

runner = ApiExperimentRunner(client=client, config=config)
runner.setup()
all_rows = runner.run_comparison()

## 4. Inspect summaries (and persist to Drive if you like)

In [ ]:
import json, glob
for path in sorted(glob.glob('results/*_summary.json')):
    print(path)
    print(json.dumps(json.loads(open(path).read())['metrics'], indent=2))
    print()

# Optional: persist to Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r results /content/drive/MyDrive/chat_dt_results